# This notebook intends to address task1 -- Capacity Plan
the task1 has two parts:

1. Calcualte hoursly demand prediction for each parts via breaking down annual pull demand from client A, B into 
2. Transform demand for production into capacity requirements for equipments, personnel via formulating and solving an IP problem for best set up of resources.

__Note that the planning of resources can be changed in later tasks if for the need from layout design__

In [68]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus

## Demand Calculation

the information on annual demand turns out to be redundant only set to help conceptualize the scenario

In [69]:
# parameter settings:

products = ['A1', 'A2', 'A3', 'B1', 'B2']

# weekly demand 
demand_data = {
    'Weekly_Mean': [
        962,  
        1923, 
        2500, 
        1154, 
        1538  
    ],
    'Weekly_CV': [0.15, 0.2, 0.2, 0.12, 0.18] 
}

demand_df = pd.DataFrame(demand_data, index=products) 

# the BOM of requirements of each Products (A, B) from each Parts (P1~P20)
bom_df = pd.read_csv('../Data/BOM.csv', index_col=0).T # for the later matrix multiplication dim align
print('BOM of product to parts:\n', bom_df.head())


BOM of product to parts:
     A1  A2  A3  B1  B2
P1   1   2   4   4   1
P2   4   0   2   2   0
P3   0   0   0   4   1
P4   0   1   2   2   0
P5   0   1   2   0   0


since clients require 99.5% of service level for replenishment from FaMoSa, the target capacity is set to be 99.5% percentile of the demand

there is limited information on familty of distribution for demand, with following justifications:
- the demand sample has a moderately large size ($\mu > 30$)
- the variation is mild ($\sigma < \mu$)


thus we assume the demand distribution follows normal

In [70]:
# calculate the target capacity for products
Z_score = norm.ppf(0.995)

demand_df['Weekly_std'] = demand_df['Weekly_Mean'] * demand_df['Weekly_CV']
# round up to maintain integer
demand_df['Product_Capacity'] = np.ceil(demand_df['Weekly_Mean'] + demand_df['Weekly_std']*Z_score)


# Aggregate to parts demand
part_demand = pd.DataFrame()
part_demand['Part_Weekly_Capacity'] = bom_df.dot(demand_df['Product_Capacity'])

part_demand['Part_Weekly_Capacity'] = np.ceil(part_demand['Part_Weekly_Capacity']) 

# Fix: Strip whitespace from the index
part_demand.index = part_demand.index.str.strip()

print('weekly capacity for each product:\n', demand_df['Product_Capacity'])
print('\nweekly capacity for each part:\n', part_demand)

weekly capacity for each product:
 A1    1334.0
A2    2914.0
A3    3788.0
B1    1511.0
B2    2252.0
Name: Product_Capacity, dtype: float64

weekly capacity for each part:
      Part_Weekly_Capacity
P1                30610.0
P2                15934.0
P3                 8296.0
P4                13512.0
P5                10490.0
P6                 5838.0
P7                20664.0
P8                 6044.0
P9                13404.0
P10               18289.0
P11               11676.0
P12               14344.0
P13                7172.0
P14               26758.0
P15                3022.0
P16               18066.0
P17                7172.0
P18               17258.0
P19               29276.0
P20               21725.0


## Capacity Planning

after validating the demand for each part, in this case each parts are produced through a series of processes naming A, B, C...M. Meanwhile, some machine can do one step and some can do multiple and that is showcased via their naming (e.g. machien AB can do A and B step)

Thus we formulte the problem as followsings:

### Sets
- $P$: set of products (parts), index $p$
- $S$: set of process steps, index $s$
- $M$: set of machine types (or pools), index $m$

### Parameters

$D_{ps} \ge 0$: required processing units for product $p$ at step $s$ in the planning period

$\eta_{ps} \ge 0$: productivity of machines on $Step_ps$, while machines do not casue differences in time of processing, there is no $m$ dimension

$$\eta_{ps} = \frac{60}{t_{ps}}$$

$A_m > 0$: available hours per installed machine of type $m$ in the period (calendar hours $\times$ planned shifts)

$\mathrm{OEE}_m \in (0,1]$: effective efficiency factor (availability $\times$ performance $\times$ quality) for type $m$

$K_m \ge 0$: installation/fixed cost per machine of type $m$ 

$C^{\text{lab}}_m \ge 0$: labor cost per machine-hour used on type $m$

$E_{ms} \in \{0,1\}$: eligibility indicator; $E_{ms} = 1$ if type $m$ can perform step $s$, else $0$

for efficiency, we define effective per-machine hours:
$$A_m^{eff} = A_m * \mathrm{OEE}$$

### Decision Vriables
- $y_m \in \mathbb{Z}_{+}$: number of machines of type $m$ to install/plan
- $x_{mps} \ge 0$: machine-hours of type $m$ allocated to product $p$ at step $s$

### Objective
$$\min Z = \sum_{m \in M} K_m y_m + \sum_{m \in M} C_m^{\text{lab}} \left( \sum_{p \in P} \sum_{a \in S} x_{mpa} \right)$$

### Constraints

- Unite demand based satisfaction of each step:
$$\sum_m \eta_{ps} * X_{mps} \ge D_{ps} \quad \forall p \in P, s \in S$$

- Capacity coupling by machine type:
$$\sum_{p \in P} \sum_{s \in S} x_{mps} \leq A_m^{\text{eff}} y_m \quad \forall m \in M$$

- Eligibility (step-machine):
$$x_{mps} = 0 \quad \text{whenever } E_{ms} = 0$$

- Variable domains:
$$x_{mps} \geq 0 \quad \forall m, p, s; \qquad y_m \in \mathbb{Z}_+ \quad \forall m$$

__Note that the horizon of timeframe of all the numerical values are within a wek__

In [71]:
# load parameters

path = "../Data"  # Go up one level then into Data
step = pd.read_csv(f"{path}/Steps.csv", index_col=0)
timecost = pd.read_csv(f"{path}/Timecost.csv", index_col=0)
 
print(step.head())
print("\n\n", timecost.head())

     Step 1 Step 2 Step 3 Step 4 Step 5 Step 6 Step 7
Part                                                 
P1        B      A      B      C      D      I      J
P2        A      C      D      H      J    NaN    NaN
P3        B      D      C      I      J    NaN    NaN
P4        A      B      D      G      H    NaN    NaN
P5        B      C      C      I    NaN    NaN    NaN


       Step 1  Step 2  Step 3  Step 4  Step 5  Step 6  Step 7
Part                                                        
P1      2.50    1.00    2.50    0.50    2.50    1.25     2.5
P2      1.25    0.50    2.50    1.00    2.50     NaN     NaN
P3      1.75    3.00    0.75    1.50    2.50     NaN     NaN
P4      1.00    2.00    3.00    0.25    1.25     NaN     NaN
P5      1.50    0.75    3.50    1.75     NaN     NaN     NaN


In [72]:
# calculate the productivity matrix

productivity = 60 / timecost

### calculate the effective working time
- 90% efficiency and 98% reliability of all equipment, with 100% quality at each operation
 
- all the machines are working one or two 8hr shift perday and 5 days a week 

Here we assume one shifts per day for larger tolerance in the planning. However, this can be changed

In [73]:
OEE = 0.9 * 0.98
A = 5 * 8 
A_eff = A * OEE

### Cost Related

- Intallation cost can be found for each machine in [Equipment Information](../Data/Equipment.csv)

- Labor cost are related to configurations of workers to each machine and workers salaries

there are two types of workers: `C1`, `C2`, `C3` with different paticipation in each machine (e.g. a C2 worker need to put 1/4 of his time in this machine A)
$$C_m^{lab} = \sum_{l \in \{C1, C2, C3\}} P_{lm} S_l$$

where:
- $P_{lm}$ for each worker's participation in the machine
- $S_l$ for each worker's hourly salary
- $l \in \{C1, C2\}$ represents the labor types (C1, C2, C3)

In [74]:
# load the parameters
machine = pd.read_csv("../Data/Equipment.csv", index_col='Equipment', usecols=['Equipment', 'Installed price', 'C1', 'C2', 'C3'])

K_m = machine.loc[:, 'Installed price']

salary = pd.Series([40, 75, 100], index=['C1', 'C2', 'C3'])
salary_df = pd.DataFrame(salary, columns=['Salary']) # note the columns require list 

machine_labor = machine[['C1', 'C2', 'C3']]
labor_cost = (machine_labor.dot(salary_df).T).T
print(labor_cost.head())

           Salary
Equipment        
A           58.75
B           58.75
C           58.75
D           58.75
AB          58.75


## Solver Calling

In [75]:
from capacity_planning_solver import CapacityPlanningSolver

# transform data structure first
part_demand_dict = part_demand['Part_Weekly_Capacity'].to_dict()
K_m_dict= K_m.to_dict()
C_m_lab_dict = labor_cost['Salary'].to_dict()


# call solver class
solver = CapacityPlanningSolver()

# create eligibility matrix
all_steps = solver._extract_all_steps(step)
eligibility = {}
for machine in machine.index:
    eligibility[machine] = {}
    for s in all_steps:
        # Machine 'AB' can do steps 'A' and 'B'
        # Machine 'ABCD' can do steps 'A', 'B', 'C', 'D'
        eligibility[machine][s] = 1 if s in machine else 0

In [76]:
# run solver
results = solver.solve(
    part_demand=part_demand_dict,
    step_df=step,
    productivity_df=productivity,
    K_m=K_m,
    C_m_lab=C_m_lab_dict,
    A_eff=A_eff,
    eligibility=eligibility,
    verbose=True
)

CAPACITY PLANNING OPTIMIZATION

Building optimization model...
Model Statistics:
  - Variables: 6786
  - Constraints: 5990
  - Demand-step combinations: 84

Solving MIP model...
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/hty/Documents/Working Projects/ISYE6202/venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/c64743c1dccf4852aedd3985309ecff1-pulp.mps -sec 300 -timeMode elapsed -branch -printingOptions all -solution /tmp/c64743c1dccf4852aedd3985309ecff1-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 5995 COLUMNS
At line 25754 RHS
At line 31745 BOUNDS
At line 31772 ENDATA
Problem MODEL has 5990 rows, 6786 columns and 12920 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+100 to 300
Option for timeMode changed from cpu to elapsed
Continuous objective value is 4.12085e+08 - 0.00 seconds
Cgl0003I 0 fixed, 23 tightened bounds, 0 strengthened rows, 0 su

In [77]:
# get the distribution of time 

print(f"The machine allocations:", solver.get_machine_allocation())

The machine allocations:    Machine  Quantity  Installation_Cost  Utilization_%   Hours_Used  \
0        A        32          4800000.0     100.000000  1128.960000   
1       AC         1           280000.0     100.000000    35.280000   
2        B        67         13400000.0      99.488035  2351.658370   
3        C        25          6250000.0     100.000000   882.000000   
4       CD         1           350000.0      82.242064    29.015000   
5        D        94         28200000.0     100.000001  3316.320040   
6        E        66         26400000.0      99.733160  2322.266680   
7        F        63         25200000.0     100.000000  2222.640003   
8       FG         1           450000.0      51.892007    18.307500   
9        G        53         21200000.0     100.000000  1869.839997   
10       H       100        100000000.0      99.114466  3496.758350   
11       I       104         52000000.0      99.275962  3642.554160   
12       J       154        115500000.0      99.8902